> 在线讲义：[第 18 章 · 上真机：执行器网络、动作修正量与 sim2sim 检查](https://wikieden.github.io/robot-motion-rl/lessons/18-deploy.html) · 本书《机器人运控与强化学习》

# 第 18 章 · 上真机：执行器网络、动作修正量与 sim2sim 检查

配套讲义：仓库里 `robot-motion-rl/notes/lessons/18-deploy.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（仿真和真机的公式是编的）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第三步 差距 | sim vs real | `sim_step`、`real_step` | a = 1：1.00 vs 0.75 |
| 第三步 修正 | 最小二乘 | `fit_delta_action([0, …, 1])` | Δa = −0.2a − 0.05 |
| 第四步 延迟 | ⌈延迟/周期⌉ | `delay_steps(ms)` | 5、20、30 ms → 1、1、2 |

玩具系统（编的）：仿真 x' = x + a；真机电机弱 20% 还有 0.05 的摩擦：x' = x + 0.8·a − 0.05。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `sim_step`**：仿真里的一步。

In [ ]:
def sim_step(x, a):
    """仿真里的一步。"""
    return x + a

**函数 `real_step`**：真机的一步（编的偏差）。

In [ ]:
def real_step(x, a):
    """真机的一步（编的偏差）。"""
    return x + 0.8 * a - 0.05

**函数 `fit_delta_action`**：ASAP 思路：学 Δa(a)，让 sim(a + Δa) 等于 real(a)。这里用线性模型 Δa = c1·a + c0 做最小二乘。

In [ ]:
def fit_delta_action(actions):
    """ASAP 思路：学 Δa(a)，让 sim(a + Δa) 等于 real(a)。这里用线性模型 Δa = c1·a + c0 做最小二乘。"""
    a = np.asarray(actions, float)
    target = np.array([real_step(0.0, v) - sim_step(0.0, v) for v in a])
    design = np.stack([a, np.ones_like(a)], axis=1)
    c1, c0 = np.linalg.lstsq(design, target, rcond=None)[0]
    return float(c1), float(c0)

**函数 `delay_steps`**：延迟相当于晚几个控制周期（向上取整）。

In [ ]:
def delay_steps(latency_ms, control_period_ms=20):
    """延迟相当于晚几个控制周期（向上取整）。"""
    return -(-latency_ms // control_period_ms)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 现实差距")
for a in (0.0, 0.5, 1.0):
    print(f"a = {a}：仿真走 {sim_step(0, a):.2f}，真机走 {real_step(0, a):.2f}")

In [ ]:
print("\n第三步 学动作修正量")
c1, c0 = fit_delta_action([0.0, 0.25, 0.5, 0.75, 1.0])
print(f"Δa = {c1:.3f}·a + {c0:.3f}")
a = 1.0
print(f"a = 1：仿真里执行 a + Δa = {a + c1 * a + c0:.2f}，走 {sim_step(0, a + c1 * a + c0):.2f}（真机 {real_step(0, a):.2f}）")

In [ ]:
print("\n第四步 延迟")
for ms in (5, 20, 30):
    print(f"延迟 {ms} ms，控制周期 20 ms → 晚 {delay_steps(ms)} 个周期")

## 3. 代码里才看得到的

- 玩具里仿真是 $x' = x + a$，所以「要补的位移」和「要补的动作」数值相同；真实的 ASAP 修正模型是一个以状态和动作为输入的神经网络，用强化学习训练，让仿真轨迹贴近真机轨迹。

## 4. 自己改一改

- 给 `real_step` 加一点噪声，看拟合出来的 c1、c0 偏多少。
- 让真机的摩擦和速度有关（比如 −0.05 − 0.1·x），看线性修正还够不够用。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 模拟器学会真车开：电机、参数、延迟、传感四处差；
> 电机写不准就学一个网络，塞回仿真里；
> 方向盘打三十真车转二十四，学个 Δa 把手感对齐，练完再拿掉；
> 换个试车场先跑一圈，接口观测随机延迟安全，逐项核对再上机。